# Datathon Passos Mágicos — Análise de Negócio (Perguntas 1 a 11)

Este notebook responde as dores de negócio com análise descritiva + analítica.
Ajuste os textos finais com os números reais que o notebook vai gerar na sua máquina.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from passos_data import add_phase_order, make_long_for_trend
from plots_passos import (
    plot_barras_defasagem_por_ano, plot_serie_indicadores, plot_scatter_relacao,
    plot_matriz_correlacao, plot_box_indicador_por_fase
)

pd.set_option("display.max_columns", 120)

df = pd.read_csv(PROJECT_ROOT / "data_processed" / "pede_consolidado.csv")
df = add_phase_order(df)
for c in ["inde","ida","ieg","iaa","ips","ipp","ipv","ian","defasagem"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
display(df.head())

## Helpers

In [ ]:
def resumo_media_por_ano(df, col):
    return (df.groupby("ano_referencia", as_index=False)[col]
              .mean()
              .rename(columns={col: f"media_{col}"}))

def resumo_media_por_ano_fase(df, col):
    if "fase" not in df.columns:
        return pd.DataFrame()
    return (df.groupby(["ano_referencia", "fase"], as_index=False)[col]
              .mean()
              .sort_values(["ano_referencia", "fase"]))

def corr_safe(df, a, b):
    x = pd.to_numeric(df[a], errors="coerce")
    y = pd.to_numeric(df[b], errors="coerce")
    m = x.notna() & y.notna()
    if m.sum() < 3:
        return np.nan
    return x[m].corr(y[m])

def tabela_correlacoes(df, pares):
    rows = []
    for a, b in pares:
        if a in df.columns and b in df.columns:
            rows.append({"var_1": a, "var_2": b, "corr_pearson": corr_safe(df, a, b)})
    return pd.DataFrame(rows).sort_values("corr_pearson", key=lambda s: s.abs(), ascending=False)

## 1) Adequação do nível (IAN) — perfil geral de defasagem e evolução

In [ ]:
if "defasagem" in df.columns:
    perfil = (df.assign(categoria_defasagem=np.select(
                [df["defasagem"] <= -2, df["defasagem"] == -1, df["defasagem"] >= 0],
                ["Severa", "Moderada", "Adequado/Acima"],
                default="Sem dado"))
               .groupby(["ano_referencia","categoria_defasagem"], as_index=False)
               .size()
               .rename(columns={"size":"qtd"}))
    display(perfil)

    plot_barras_defasagem_por_ano(df, outpath="outputs/figs/q1_perfil_defasagem_por_ano.png")
else:
    print("Sem coluna 'defasagem' — usar IAN como proxy.")
    if "ian" in df.columns:
        display(resumo_media_por_ano(df, "ian"))

## 2) IDA — desempenho acadêmico melhora, estagna ou cai?

In [ ]:
if "ida" in df.columns:
    ida_ano = resumo_media_por_ano(df, "ida")
    display(ida_ano)

    ida_ano_fase = resumo_media_por_ano_fase(df, "ida")
    display(ida_ano_fase.head(20))

    long_df = make_long_for_trend(df)
    plot_serie_indicadores(long_df, indicadores=["ida","inde"], outpath="outputs/figs/q2_tendencia_ida_inde.png")

## 3) IEG vs IDA/IPV — engajamento tem relação com desempenho?

In [ ]:
pares_q3 = [("ieg","ida"),("ieg","ipv"),("ieg","inde")]
display(tabela_correlacoes(df, pares_q3))

if all(c in df.columns for c in ["ieg","ida"]):
    plot_scatter_relacao(df, "ieg", "ida", hue="ano_referencia", outpath="outputs/figs/q3_ieg_vs_ida.png")
if all(c in df.columns for c in ["ieg","ipv"]):
    plot_scatter_relacao(df, "ieg", "ipv", hue="ano_referencia", outpath="outputs/figs/q3_ieg_vs_ipv.png")

## 4) IAA — autoavaliação é coerente com IDA e IEG?

In [ ]:
pares_q4 = [("iaa","ida"),("iaa","ieg"),("iaa","inde")]
display(tabela_correlacoes(df, pares_q4))

if all(c in df.columns for c in ["iaa","ida"]):
    plot_scatter_relacao(df, "iaa", "ida", hue="ano_referencia", outpath="outputs/figs/q4_iaa_vs_ida.png")

## 5) IPS — padrões psicossociais antecedem quedas?

In [ ]:
# Proxy 1: relação com piora no próximo ano (se houver histórico longitudinal por RA)
if all(c in df.columns for c in ["ips", "piora_defasagem_prox_ano"]):
    tmp = df[["ips","piora_defasagem_prox_ano","ano_referencia"]].copy()
    tmp["ips"] = pd.to_numeric(tmp["ips"], errors="coerce")
    tmp["piora_defasagem_prox_ano"] = pd.to_numeric(tmp["piora_defasagem_prox_ano"], errors="coerce")
    display(tmp.groupby("piora_defasagem_prox_ano", as_index=False)["ips"].mean())

# Proxy 2: correlação com indicadores de desempenho/engajamento
pares_q5 = [("ips","ida"),("ips","ieg"),("ips","ipv"),("ips","defasagem")]
display(tabela_correlacoes(df, pares_q5))

## 6) IPP vs IAN — avaliações psicopedagógicas confirmam ou contradizem a defasagem?

In [ ]:
pares_q6 = [("ipp","ian"),("ipp","defasagem"),("ipp","ida")]
display(tabela_correlacoes(df, pares_q6))

if all(c in df.columns for c in ["ipp","ian"]):
    plot_scatter_relacao(df, "ipp", "ian", hue="ano_referencia", outpath="outputs/figs/q6_ipp_vs_ian.png")

## 7) IPV — quais comportamentos mais influenciam o ponto de virada?

In [ ]:
pares_q7 = [("ipv","ida"),("ipv","ieg"),("ipv","iaa"),("ipv","ips"),("ipv","ipp"),("ipv","ian")]
display(tabela_correlacoes(df, pares_q7))

corr_cols = [c for c in ["ipv","ida","ieg","iaa","ips","ipp","ian"] if c in df.columns]
if len(corr_cols) >= 2:
    plot_matriz_correlacao(df, corr_cols, outpath="outputs/figs/q7_correlacao_ipv.png")

## 8) Multidimensionalidade — combinações que elevam INDE

In [ ]:
if "inde" in df.columns:
    # Faixas por quartil para leitura gerencial
    tmp = df.copy()
    for c in ["ida","ieg","ips","ipp"]:
        if c in tmp.columns:
            tmp[f"{c}_faixa"] = pd.qcut(pd.to_numeric(tmp[c], errors="coerce"), q=4, duplicates="drop")
    cols_combo = [c for c in ["ida_faixa","ieg_faixa","ips_faixa","ipp_faixa"] if c in tmp.columns]
    if cols_combo:
        combo = (tmp.groupby(cols_combo, dropna=False, as_index=False)["inde"]
                   .mean()
                   .sort_values("inde", ascending=False))
        display(combo.head(15))

## 9) Previsão de risco com ML

O notebook **02_modelo_risco_ml.ipynb** implementa:
- definição do target de risco (defasagem atual e/ou piora futura)
- treino e validação temporal
- curva ROC/PR e calibração
- probabilidade de risco por aluno
- importância das variáveis

## 10) Efetividade do programa por fase

In [ ]:
for indicador in ["inde","ida","ieg","ipv","ian"]:
    if indicador in df.columns and "fase" in df.columns:
        print(f"\n### {indicador.upper()} por fase/ano")
        display(resumo_media_por_ano_fase(df, indicador).head(50))
        try:
            plot_box_indicador_por_fase(df, indicador, outpath=f"outputs/figs/q10_box_{indicador}_por_fase.png", show=False)
            print(f"Gráfico salvo: outputs/figs/q10_box_{indicador}_por_fase.png")
        except Exception as e:
            print("Erro ao gerar boxplot:", e)

## 11) Insights extras e sugestões

Sugestões de análises criativas para fortalecer o case:
1. **Transição entre pedras** (Markov simplificado): probabilidade de evolução Quartzo→Ágata→Ametista→Topázio.
2. **Coortes de entrada** (`ano_ingresso`): comparar crescimento de INDE/IDA por coorte.
3. **Risco por turma** (sem expor dados pessoais): ranking de turmas com maior concentração de risco.
4. **Segmentação de apoio**: cluster leve com indicadores (ex.: alto IEG e baixo IDA vs baixo IEG e baixo IAA).
5. **Gap percepção x realidade**: `IAA - normalização(IDA)` para detectar super/subestimação.
6. **Lead indicators**: testar se IPS/IEG de um ano explicam IDA/IPV no ano seguinte.

## Fechamento para apresentação gerencial (modelo de narrativa)

- **Situação atual**: perfil de defasagem e distribuição de risco por ano/fase  
- **Evidência de impacto**: evolução de IDA/INDE/IEG/IPV ao longo de 2022–2024  
- **Fatores críticos**: relações entre engajamento, psicossocial e desempenho  
- **Predição**: modelo de risco para antecipar intervenção  
- **Ação recomendada**: trilhas de acompanhamento por perfil de aluno